# 빠른 언더필 판정 모델: 질감 게이트 + DINOv2 고정 특징 + 로지스틱 회귀

신경망은 학습하지 않습니다. 공개 사전학습 DINOv2를 고정해 사진마다 특징을 한 번 뽑고, 로지스틱 회귀만 학습합니다.

**구성**
- 전처리: 고정 크롭 [0, 250, 1000, 1250] → 518×518 축소(bilinear, antialias) → ImageNet 정규화
- 게이트: `filament_gate.py` 질감 기준, 출력물 면적 0.1 미만이면 보류 (학습에서도 제외)
- 특징: DINOv2 ViT-B/14 (`vit_base_patch14_dinov2.lvd142m`), CLS 768 + 조각 평균 768 = 1,536개
- 분류: 표준화 + 로지스틱 회귀 (C=1.0), 결함 확률 0.5 이상이면 언더필
- 평가: 시간 블록 5-fold (`folds_all4000.csv`, 두현님 분할표와 같음)

**Drive `산공종설/`에 둘 파일**: `underfill_originals_4000.zip`, `filament_gate.py`, `folds_all4000.csv`

**실행**: 런타임 > 런타임 유형 변경 > GPU → 1번부터 차례로. 오래 걸리는 건 6번(특징 추출) 한 번뿐이고 나머지는 몇 초~몇 분입니다. 결과는 `산공종설/results/fast_model/`에 저장되고, 끊겨도 저장된 값은 다시 계산하지 않습니다.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > GPU"

## 2. Drive 연결 + 파일 확인

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os, glob, sys, shutil
DRIVE_DIR = "/content/drive/MyDrive/산공종설"   # 폴더 이름 다르면 여기만 바꾸기
OUT = f"{DRIVE_DIR}/results/fast_model"
os.makedirs(OUT, exist_ok=True)
for f in ("underfill_originals_4000.zip", "filament_gate.py", "folds_all4000.csv"):
    print("OK  " if os.path.exists(f"{DRIVE_DIR}/{f}") else "없음", f)

## 3. 원본 풀기 + 게이트 불러오기
마지막 줄에 `4000`이 나와야 합니다.

In [ ]:
if len(glob.glob("/content/orig/**/*.jpg", recursive=True)) < 4000:
    !cp "{DRIVE_DIR}/underfill_originals_4000.zip" /content/
    !unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
shutil.copy(f"{DRIVE_DIR}/filament_gate.py", "/content/filament_gate.py")
sys.path.insert(0, "/content")
import importlib, filament_gate as fg
importlib.reload(fg); fg._self_check()
print(len(glob.glob("/content/orig/**/*.jpg", recursive=True)))

## 4. 설정 + 사진 목록
분할표는 사진 4,000장 × fold 5개 = 20,000줄이고, 각 사진은 한 fold에서만 시험(`val`)으로 쓰입니다.

In [ ]:
import numpy as np, pandas as pd, cv2, time
from pathlib import Path

ROI = fg.ROI
GATE_T = fg.THRESHOLD     # 0.1
PROB_T = 0.5              # 결함 판정 기준
IMG = 518                 # DINOv2는 14의 배수
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

folds = pd.read_csv(f"{DRIVE_DIR}/folds_all4000.csv", encoding="utf-8-sig")
folds["source"] = folds.source.str.replace("\\", "/", regex=False)
imgs = (folds[folds.role == "val"][["source", "label", "label_id", "frame_index", "fold"]]
        .rename(columns={"fold": "test_fold"}).sort_values("source").reset_index(drop=True))
src_idx = {s: i for i, s in enumerate(imgs.source)}
print(len(imgs), imgs.label.value_counts().to_dict())

## 5. 게이트 값 (2분 안팎, 한 번만)
게이트 확인 노트북을 이미 돌렸다면 `results/gate_check/gate_values.csv`를 그대로 가져옵니다.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
gate_file = f"{OUT}/gate_values.csv"
other = f"{DRIVE_DIR}/results/gate_check/gate_values.csv"
if not os.path.exists(gate_file) and os.path.exists(other):
    shutil.copy(other, gate_file); print("게이트 확인 노트북 결과 재사용")
if os.path.exists(gate_file):
    gv = pd.read_csv(gate_file, encoding="utf-8-sig")
else:
    t0 = time.time()
    def one(s): return fg.filament_ratio(fg.crop_roi(cv2.imread(f"/content/orig/{s}")), bgr=True)
    with ThreadPoolExecutor(4) as ex:
        vals = list(ex.map(one, imgs.source))
    gv = pd.DataFrame({"source": imgs.source, "new": np.round(vals, 4)})
    gv.to_csv(gate_file, index=False, encoding="utf-8-sig"); print(f"완료: {time.time()-t0:.0f}초")
imgs = imgs.drop(columns=["new"], errors="ignore").merge(gv[["source", "new"]], on="source", how="left")
assert imgs.new.notna().all()
imgs["passed"] = imgs.new >= GATE_T
print("게이트 보류:", (~imgs.passed).groupby(imgs.label).sum().to_dict(), "| 통과", imgs.passed.sum(), "장")

## 6. DINOv2 특징 추출 (한 번만)
4,000장 전부 특징을 뽑아 `feats_dinov2.npy`로 저장합니다. 이미 있으면 불러오기만 합니다. 처음 실행 때 가중치를 인터넷에서 받습니다(HF_TOKEN 경고는 무시해도 됨). 걸린 시간이 마지막에 출력됩니다.

In [ ]:
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision.transforms.v2 import functional as TF

def to_input(bgr):
    # 원본(BGR) -> 고정 크롭 -> RGB uint8 -> 518 축소 -> 0~1 -> ImageNet 정규화
    crop = fg.crop_roi(bgr)[:, :, ::-1]
    t = torch.from_numpy(np.ascontiguousarray(crop)).permute(2, 0, 1)
    t = TF.resize(t, [IMG, IMG], interpolation=v2.InterpolationMode.BILINEAR, antialias=True)
    return TF.normalize(t.float() / 255.0, MEAN, STD)

dev = "cuda"
dn = timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=True, num_classes=0).eval().to(dev)
NPT = dn.num_prefix_tokens

@torch.no_grad()
def extract(x):
    with torch.autocast("cuda", dtype=torch.float16):
        tok = dn.forward_features(x.to(dev)).float()
    return torch.cat([tok[:, 0], tok[:, NPT:].mean(1)], 1).cpu()   # CLS + 조각 평균

feat_file = f"{OUT}/feats_dinov2.npy"
if os.path.exists(feat_file):
    X = np.load(feat_file); print("저장된 특징 불러옴")
else:
    class Imgs(Dataset):
        def __len__(self): return len(imgs)
        def __getitem__(self, i): return to_input(cv2.imread(f"/content/orig/{imgs.source[i]}"))
    out, t0 = [], time.time()
    for b, x in enumerate(DataLoader(Imgs(), batch_size=16, num_workers=2)):
        out.append(extract(x))
        if b % 50 == 0: print(f"{(b+1)*16:5d} / {len(imgs)}장 ({time.time()-t0:.0f}초)")
    X = torch.cat(out).numpy(); np.save(feat_file, X)
    print(f"완료: {time.time()-t0:.0f}초")
print(X.shape)

## 7. 5-fold 평가 (몇 초)
fold마다 로지스틱 회귀를 학습하고 평가합니다. 비교용으로 게이트 없는 결과도 같이 냅니다. 지표는 **판정한 사진 기준**이고, `판정비율`을 같이 봐야 합니다.

참고(같은 분할, 두현님 2026-10-02 보고서, 시드 3개 평균, 기존 게이트로 3,951장): ConvNeXt-Tiny 정확도 0.931, AUROC 0.991, 오탐 219, 미탐 52 / ResNet18 정확도 0.922, AUROC 0.973

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def fit(Xtr, ytr):
    sc = StandardScaler().fit(Xtr)
    return sc, LogisticRegression(max_iter=5000).fit(sc.transform(Xtr), ytr)

def cv_prob(use_gate):
    p = np.full(len(imgs), np.nan); t0 = time.time()
    for k in range(5):
        f = folds[folds.fold == k]
        tr, te = f[f.role == "train"], f[f.role == "val"]
        itr, ite = tr.source.map(src_idx).values, te.source.map(src_idx).values
        if use_gate:
            itr, ite = itr[imgs.passed.values[itr]], ite[imgs.passed.values[ite]]
        sc, clf = fit(X[itr], imgs.label_id.values[itr])
        p[ite] = clf.predict_proba(sc.transform(X[ite]))[:, 1]
    return p, time.time() - t0

def summarize(p):
    y = imgs.label_id.values; ok = ~np.isnan(p); pred = (p >= PROB_T).astype(int)
    tp = ((pred == 1) & (y == 1) & ok).sum(); fn = ((pred == 0) & (y == 1) & ok).sum()
    fp = ((pred == 1) & (y == 0) & ok).sum(); tn = ((pred == 0) & (y == 0) & ok).sum()
    acc_k = [((pred == y) & ok & (imgs.test_fold == k)).sum() / (ok & (imgs.test_fold == k)).sum() for k in range(5)]
    return {"판정비율": round(ok.mean(), 4), "AUROC": round(roc_auc_score(y[ok], p[ok]), 4),
            "정확도": round((tp + tn) / ok.sum(), 4), "언더필 재현율": round(tp / (tp + fn), 4),
            "정상 재현율": round(tn / (tn + fp), 4), "오탐": int(fp), "미탐": int(fn),
            "fold별 정확도": " ".join(f"{a:.3f}" for a in acc_k)}

prob = {}
rows = []
for name, g in (("게이트 없음", False), (f"질감 게이트 {GATE_T}", True)):
    prob[name], sec = cv_prob(g)
    rows.append({"조건": name, **summarize(prob[name]), "학습+평가 시간(초)": round(sec, 1)})
summary = pd.DataFrame(rows)
summary.to_csv(f"{OUT}/summary.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.width", 220)
print(summary.to_string(index=False))

## 8. 사진별 결과 CSV + 미탐·오탐 썸네일
- `predictions.csv`: 사진 한 줄씩. 정답, 시험 fold, 게이트 값, 결함 확률, 결과(정답/미탐/오탐/보류). 엑셀에서 열림
- `thumbs/FN_p01.png`, `thumbs/FP_p01.png` …: 게이트 적용 결과에서 틀린 사진을 24장씩. 빨간색 = 게이트가 출력물로 판단한 영역

In [ ]:
import matplotlib.pyplot as plt
GKEY = f"질감 게이트 {GATE_T}"
p = prob[GKEY]; y = imgs.label_id.values; pred = (p >= PROB_T).astype(int)
res = np.where(np.isnan(p), "보류", np.where(pred == y, "정답", np.where(y == 1, "미탐", "오탐")))
pd.DataFrame({"사진": imgs.source, "정답": imgs.label, "시험fold": imgs.test_fold, "프레임": imgs.frame_index,
              "게이트값": imgs.new, "확률": np.round(p, 4), "결과": res,
              "확률_게이트없음": np.round(prob["게이트 없음"], 4)}).to_csv(f"{OUT}/predictions.csv", index=False, encoding="utf-8-sig")
print(pd.Series(res).value_counts().to_dict())

TD = f"{OUT}/thumbs"; os.makedirs(TD, exist_ok=True)
for kind, tag in (("미탐", "FN"), ("오탐", "FP")):
    idx = np.where(res == kind)[0]; idx = idx[np.argsort(imgs.frame_index.values[idx])]
    for pg in range(0, len(idx), 24):
        chunk = idx[pg:pg + 24]
        fig, axs = plt.subplots(4, 6, figsize=(15, 11))
        for a in axs.flat: a.axis("off")
        for a, i in zip(axs.flat, chunk):
            a.imshow(fg.overlay(fg.crop_roi(cv2.imread(f"/content/orig/{imgs.source[i]}")), bgr=True))
            a.set_title(f"{imgs.label[i][:9]} #{imgs.frame_index[i]}  p={p[i]:.2f}\ngate={imgs.new[i]:.2f}", fontsize=8)
        fig.suptitle(f"{tag} {pg+1}-{pg+len(chunk)} / {len(idx)}")
        plt.tight_layout(); plt.savefig(f"{TD}/{tag}_p{pg//24+1:02d}.png", dpi=80)
        if pg == 0: plt.show()
        plt.close(fig)
    print(tag, len(idx), "장")

## 9. 최종 모델 저장
게이트를 통과한 **전체** 사진으로 로지스틱 회귀를 한 번 학습해 `fast_model.joblib`로 저장합니다. 새 사진 판정(10번)에 쓰는 모델입니다. 평가 점수는 7번(5-fold)을 기준으로 보세요.

In [ ]:
import joblib
ia = np.where(imgs.passed.values)[0]
t0 = time.time(); sc, clf = fit(X[ia], imgs.label_id.values[ia])
bundle = {"scaler": sc, "clf": clf, "backbone": "vit_base_patch14_dinov2.lvd142m", "img": IMG, "roi": ROI,
          "gate_threshold": GATE_T, "prob_threshold": PROB_T, "n_train": len(ia)}
joblib.dump(bundle, f"{OUT}/fast_model.joblib")
print(f"학습 {len(ia)}장, {time.time()-t0:.1f}초 → {OUT}/fast_model.joblib")

## 10. 새 사진 판정 + 속도 측정
`judge(경로)`가 사진 한 장을 게이트 → 특징 → 확률 → 판정까지 처리합니다. `JUDGE_DIR`에 사진 폴더를 넣으면 폴더 전체를 판정해 `judged.csv`로 저장합니다(파일 이름 순서).

아래 예시는 동작과 속도 확인용으로 원본 몇 장을 넣습니다. **이 사진들은 9번 최종 모델의 학습에 들어간 사진이라 정확도 확인용이 아닙니다.** 속도는 사진 읽기를 포함한 한 장 처리 시간(첫 장 제외 중앙값)입니다.

In [ ]:
def judge(path, model=bundle):
    t0 = time.perf_counter()
    bgr = cv2.imread(str(path))
    ok, r = fg.gate(fg.crop_roi(bgr), model["gate_threshold"], bgr=True)
    prob_u = None
    if ok:
        f = extract(to_input(bgr).unsqueeze(0)).numpy()
        prob_u = float(model["clf"].predict_proba(model["scaler"].transform(f))[0, 1])
    torch.cuda.synchronize()
    verdict = "보류" if prob_u is None else ("언더필" if prob_u >= model["prob_threshold"] else "정상")
    return {"사진": Path(path).name, "게이트값": round(r, 3), "확률": None if prob_u is None else round(prob_u, 4),
            "판정": verdict, "ms": round((time.perf_counter() - t0) * 1000, 1)}

JUDGE_DIR = None   # 예: f"{DRIVE_DIR}/new_frames"  (None이면 아래 예시 사진 사용)
files = sorted(glob.glob(f"{JUDGE_DIR}/*.jpg")) if JUDGE_DIR else \
        [f"/content/orig/{s}" for s in imgs.source.iloc[::200]]
out = pd.DataFrame([judge(f) for f in files])
out.to_csv(f"{OUT}/judged.csv", index=False, encoding="utf-8-sig")
print(out.head(10).to_string(index=False))
print(f"\n{len(out)}장 | 한 장 처리 중앙값 {out.ms.iloc[1:].median():.0f}ms (최대 {out.ms.iloc[1:].max():.0f}ms, GPU)")

## 11. 런타임 반납

In [ ]:
from google.colab import runtime
runtime.unassign()